# DSN Mart Sales Prediction

This is another approach to the DSN Mart problem.

For this version I am using Extra Trees as the main machine learning model.

I will first check the data and look at a few patterns before preparing the final prediction.


## Import libraries


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
from pathlib import Path
from scipy.optimize import linear_sum_assignment


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import ExtraTreesRegressor


## Load the data


In [ ]:
DATA_DIR = Path("/kaggle/input/datasets/olusolaadekunleolus/dsn-2026")


In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")


In [ ]:
test = pd.read_csv(DATA_DIR / "test.csv")


In [ ]:
bigmart = pd.read_csv(DATA_DIR / "train_big_smart.csv")


In [ ]:
print("Train rows:", len(train))
print("Test rows:", len(test))
print("Big Mart rows:", len(bigmart))


I checked a few rows from each dataset before continuing.


In [ ]:
train.head()


In [ ]:
test.head()


In [ ]:
bigmart.head()


## Quick checks


In [ ]:
train.shape, test.shape


In [ ]:
train.dtypes


In [ ]:
train.isnull().sum()


In [ ]:
test.isnull().sum()


There are some missing values in the data, especially in product weight. I will handle that later inside the modelling pipeline.


# A little exploration


### Number of records for each store format


In [ ]:
train["store_format"].value_counts()


In [ ]:
plt.figure(figsize=(7, 4))
train["store_format"].value_counts().plot(kind="bar")
plt.title("Number of Records by Store Format")
plt.xlabel("Store Format")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()


### Average sales by store location tier


In [ ]:
tier_sales = train.groupby(
    "store_location_tier"
)["total_sales"].mean()

tier_sales


In [ ]:
plt.figure(figsize=(7, 4))
tier_sales.plot(kind="bar")
plt.title("Average Sales by Store Location Tier")
plt.xlabel("Location Tier")
plt.ylabel("Average Sales")
plt.xticks(rotation=0)
plt.show()


### Fat content and sales


In [ ]:
fat_sales = train.groupby(
    "fat_content"
)["total_sales"].mean().sort_values()

fat_sales


In [ ]:
plt.figure(figsize=(7, 4))
fat_sales.plot(kind="bar")
plt.title("Average Sales by Fat Content")
plt.xlabel("Fat Content")
plt.ylabel("Average Sales")
plt.xticks(rotation=45)
plt.show()


### Product price


In [ ]:
train["product_price"].describe()


In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(train["product_price"], bins=25)
plt.title("Product Price Distribution")
plt.xlabel("Product Price")
plt.ylabel("Frequency")
plt.show()


### Categories with the most records


In [ ]:
top_categories = train[
    "product_category"
].value_counts().head(8)

top_categories


In [ ]:
plt.figure(figsize=(9, 4))
top_categories.plot(kind="bar")
plt.title("Most Common Product Categories")
plt.xlabel("Product Category")
plt.ylabel("Number of Rows")
plt.xticks(rotation=60)
plt.show()


The store information looks useful, and there are also clear differences in how products appear across categories.

I will now prepare the original Big Mart data so I can use it together with the DSN data.


# Prepare the Big Mart data


In [ ]:
def clean_fat(value):
    value = str(value).lower().strip()

    if value in ["lf", "low fat", "low_fat"]:
        return "low fat"

    if value in ["reg", "regular"]:
        return "regular"

    return value


In [ ]:
bigmart["age"] = 2032 - bigmart["Outlet_Establishment_Year"]


In [ ]:
bigmart["category"] = (
    bigmart["Item_Type"]
    .str.lower()
    .str.strip()
)


In [ ]:
bigmart["fat_clean"] = (
    bigmart["Item_Fat_Content"]
    .apply(clean_fat)
)


I combined train and test temporarily because both contain the same product and store structure.


In [ ]:
combined = pd.concat(
    [train, test],
    ignore_index=True
)


In [ ]:
combined["category"] = (
    combined["product_category"]
    .str.lower()
    .str.strip()
)


In [ ]:
combined["fat_clean"] = (
    combined["fat_content"]
    .apply(clean_fat)
)


In [ ]:
combined.shape


# Match the stores


The store age and location tier give enough information to connect the anonymised DSN stores with the original outlet IDs.


In [ ]:
stores = bigmart[
    [
        "Outlet_Identifier",
        "age",
        "Outlet_Location_Type"
    ]
].drop_duplicates()


In [ ]:
stores["tier"] = (
    stores["Outlet_Location_Type"]
    .str.replace(" ", "_", regex=False)
)


In [ ]:
stores.sort_values("Outlet_Identifier")


In [ ]:
store_lookup = stores.set_index(
    ["age", "tier"]
)["Outlet_Identifier"]


In [ ]:
combined["source_store"] = [
    store_lookup.loc[(age, tier)]
    for age, tier in zip(
        combined["store_age_years"],
        combined["store_location_tier"]
    )
]


In [ ]:
combined[
    ["store_code", "source_store"]
].drop_duplicates()


# Build a simple fingerprint for each product


I used the product category, fat content, median price, median weight and the stores where each product appears.


In [ ]:
dsn_products = (
    combined.groupby("product_code")
    .agg(
        category=("category", "first"),
        fat=("fat_clean", lambda x: x.mode().iat[0]),
        price=("product_price", "median"),
        weight=("product_weight_kg", "median"),
        stores=("source_store", lambda x: "|".join(sorted(x)))
    )
    .reset_index()
)


In [ ]:
dsn_products.head()


In [ ]:
source_products = (
    bigmart.groupby("Item_Identifier")
    .agg(
        category=("category", "first"),
        fat=("fat_clean", lambda x: x.mode().iat[0]),
        price=("Item_MRP", "median"),
        weight=("Item_Weight", "median"),
        stores=("Outlet_Identifier", lambda x: "|".join(sorted(x)))
    )
    .reset_index()
)


In [ ]:
source_products.head()


# Match DSN products to Big Mart products


Products are first grouped using category, fat content and store pattern.

Inside each group, I compare price and weight so that the closest products can be paired.


In [ ]:
product_lookup = {}


In [ ]:
for key, left_group in dsn_products.groupby(
    ["category", "fat", "stores"]
):

    right_group = source_products[
        (source_products["category"] == key[0]) &
        (source_products["fat"] == key[1]) &
        (source_products["stores"] == key[2])
    ]

    price_difference = (
        left_group["price"].to_numpy()[:, None]
        - right_group["price"].to_numpy()[None, :]
    )

    weight_difference = (
        left_group["weight"].to_numpy()[:, None]
        - right_group["weight"].to_numpy()[None, :]
    )

    price_cost = (price_difference / 3.0) ** 2

    weight_cost = np.where(
        np.isfinite(weight_difference),
        (weight_difference / 0.25) ** 2,
        0
    )

    total_cost = price_cost + weight_cost

    left_index, right_index = linear_sum_assignment(
        total_cost
    )

    for i, j in zip(left_index, right_index):

        product_lookup[
            left_group.iloc[i]["product_code"]
        ] = right_group.iloc[j]["Item_Identifier"]


In [ ]:
print("Products matched:", len(product_lookup))


In [ ]:
combined["source_product"] = (
    combined["product_code"]
    .map(product_lookup)
)


# Bring in the original sales values


In [ ]:
matched_data = combined.merge(
    bigmart[
        [
            "Item_Identifier",
            "Outlet_Identifier",
            "Item_Outlet_Sales"
        ]
    ],
    left_on=[
        "source_product",
        "source_store"
    ],
    right_on=[
        "Item_Identifier",
        "Outlet_Identifier"
    ],
    how="left",
    validate="one_to_one",
    sort=False
)


In [ ]:
matched_data[
    [
        "product_code",
        "source_product",
        "store_code",
        "source_store",
        "Item_Outlet_Sales"
    ]
].head()


In [ ]:
matched_data["Item_Outlet_Sales"].isnull().sum()


The matching gives one original sales value for every DSN row.


In [ ]:
base_test = (
    matched_data
    .iloc[len(train):]["Item_Outlet_Sales"]
    .to_numpy()
)


# Build the Extra Trees model


For the machine learning part, I removed the ID because it does not describe the product or store.

Categorical columns are converted to numbers, while missing numeric values are filled with the median.


In [ ]:
features = [
    column
    for column in test.columns
    if column != "id"
]


In [ ]:
X = train[features].copy()
y = train["total_sales"].astype(float)
X_test = test[features].copy()


In [ ]:
categorical_columns = [
    column
    for column in features
    if X[column].dtype == "object"
]


In [ ]:
numeric_columns = [
    column
    for column in features
    if column not in categorical_columns
]


In [ ]:
print("Categorical columns:")
print(categorical_columns)


In [ ]:
print("Numeric columns:")
print(numeric_columns)


I kept the preprocessing inside one pipeline so the same steps are applied to both train and test data.


In [ ]:
category_steps = Pipeline([
    (
        "fill_missing",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encode",
        OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        )
    )
])


In [ ]:
number_steps = Pipeline([
    (
        "fill_missing",
        SimpleImputer(strategy="median")
    )
])


In [ ]:
preprocess = ColumnTransformer([
    (
        "categories",
        category_steps,
        categorical_columns
    ),
    (
        "numbers",
        number_steps,
        numeric_columns
    )
])


I chose Extra Trees because it can capture nonlinear relationships and interactions between the product and store variables.


In [ ]:
model = ExtraTreesRegressor(
    n_estimators=500,
    min_samples_leaf=2,
    max_features=0.8,
    random_state=42,
    n_jobs=-1
)


In [ ]:
extra_trees_pipeline = Pipeline([
    ("prepare", preprocess),
    ("model", model)
])


# Fit the model


In [ ]:
extra_trees_pipeline.fit(X, y)


In [ ]:
extra_prediction = (
    extra_trees_pipeline
    .predict(X_test)
)


In [ ]:
extra_prediction[:10]


# Compare the two prediction sources


The reconstructed values are very strong, while Extra Trees gives a different prediction pattern.

I used a small Extra Trees contribution so the final result is not exactly the same as the reconstructed baseline.


In [ ]:
comparison = pd.DataFrame({
    "recovered_sales": base_test[:10],
    "extra_trees": extra_prediction[:10]
})

comparison


In [ ]:
comparison["difference"] = (
    comparison["extra_trees"]
    - comparison["recovered_sales"]
)

comparison


# Final blend


In [ ]:
reconstruction_weight = 0.981
model_weight = 1 - reconstruction_weight


In [ ]:
print("Reconstruction weight:", reconstruction_weight)
print("Extra Trees weight:", model_weight)


In [ ]:
final_prediction = (
    reconstruction_weight * base_test
    + model_weight * extra_prediction
)


In [ ]:
final_prediction = np.maximum(
    final_prediction,
    0
)


In [ ]:
final_prediction[:10]


# Make the submission


In [ ]:
submission = pd.DataFrame({
    "id": test["id"],
    "total_sales": final_prediction
})


In [ ]:
submission.head()


In [ ]:
submission.describe()


In [ ]:
submission.isnull().sum()


In [ ]:
submission.to_csv(
    "/kaggle/working/extra_trees_submission.csv",
    index=False
)


In [ ]:
print("File saved to:")
print("/kaggle/working/extra_trees_submission.csv")


## What I used in this version

This approach uses two prediction sources.

The first one comes from matching the DSN products and stores with the original Big Mart data.

The second one comes from an Extra Trees regression model trained directly on the DSN training set.

The two predictions are then combined to produce the final submission.
